# Data for Jiri

Generates CSV tables for 10 random samples (MLP and CNN) with the following columns per bit-width b:

- **V1** = mean width of P1 (FP model linear + correctly classifies c)
- **V2** = mean width of P2 (P1 ∩ q-model same activation pattern)
- **V2/V1** = fraction of P1 covered by P2 (shrinks as b decreases?)
- **V3_c** = mean width of P3(c) ⊆ P2 (q-model predicts correct class c)
- **V3_k0 ... V3_k9** = mean width of P3(k) for each class k (0 if empty)

Note: V3_k{c} = V3_c by definition. V3_k = 0 means P3(k) is empty (q-model never predicts k inside P2).

Output: `results/data_for_jiri/jiri_data_mlp.csv` and `jiri_data_cnn.csv`

In [ ]:
import json, glob, random, os, sys
import pandas as pd
from pathlib import Path

ROOT      = Path(os.getcwd()).parent
BITS_GRID = [4, 6, 8, 10, 12, 16]
N_SAMPLES = 10
SEED      = 42
OUT_DIR   = ROOT / 'results' / 'data_for_jiri'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT    :', ROOT)
print('OUT_DIR :', OUT_DIR)

In [ ]:
def extract_data(network, bits_grid, n_samples, seed):
    """
    Randomly select n_samples JSON result files for `network` and return
    a long-format DataFrame with one row per (sample_idx, b).

    Columns:
      sample_idx, class_c, b, V1, V2, V2_over_V1, V3_c,
      V3_k0, V3_k1, ..., V3_k9
    """
    results_dir = ROOT / 'results' / f'volumes_v3k_{network}'
    files = sorted(
        results_dir.glob('volumes_sample*.json'),
        key=lambda p: int(p.stem.split('sample')[1])
    )

    rng = random.Random(seed)
    selected = sorted(rng.sample(list(files), n_samples),
                      key=lambda p: int(p.stem.split('sample')[1]))
    print(f'{network.upper()}: selected sample indices = '
          f'{[int(p.stem.split("sample")[1]) for p in selected]}')

    rows = []
    for f in selected:
        r  = json.load(open(f))
        c  = r['class_c']
        v1 = r['width_base']          # P1 mean width — same for all b

        for b in bits_grid:
            v2  = r['widths_correct'][str(b)]   # P2 mean width
            wb  = r['widths_both'][str(b)]       # list of 10 V3_k values
            v3c = wb[c]                          # P3(c) mean width

            # Sanity checks
            assert abs(wb[c] - v3c) < 1e-9, 'V3_c mismatch'
            assert v3c <= v2 + 1e-6,         f'V3_c > V2 at sample {r["sample_idx"]}, b={b}'

            row = {
                'sample_idx' : r['sample_idx'],
                'class_c'    : c,
                'b'          : b,
                'V1'         : round(v1,  6),
                'V2'         : round(v2,  6),
                'V2_over_V1' : round(v2 / v1, 6) if v1 > 0 else float('nan'),
                'V3_c'       : round(v3c, 6),
            }
            for k in range(10):
                row[f'V3_k{k}'] = round(wb[k], 6)

            rows.append(row)

    df = pd.DataFrame(rows)
    # Reorder columns explicitly to avoid any ambiguity
    cols = (['sample_idx', 'class_c', 'b', 'V1', 'V2', 'V2_over_V1', 'V3_c']
            + [f'V3_k{k}' for k in range(10)])
    return df[cols]

In [ ]:
df_mlp = extract_data('mlp', BITS_GRID, N_SAMPLES, SEED)

out_mlp = OUT_DIR / 'jiri_data_mlp.csv'
df_mlp.to_csv(out_mlp, index=False)
print(f'Saved {len(df_mlp)} rows → {out_mlp}')
print()
print(df_mlp.to_string(index=False))

In [ ]:
df_cnn = extract_data('cnn', BITS_GRID, N_SAMPLES, SEED)

out_cnn = OUT_DIR / 'jiri_data_cnn.csv'
df_cnn.to_csv(out_cnn, index=False)
print(f'Saved {len(df_cnn)} rows → {out_cnn}')
print()
print(df_cnn.to_string(index=False))

## Sanity checks

- V3_c ≤ V2 for all rows ✓ (asserted above)
- V3_k{c} == V3_c for all rows (same polytope, two different columns)
- V1 constant across b for each sample (P1 depends only on FP model)
- V2/V1 ≤ 1 always (P2 ⊆ P1)

In [ ]:
for net, df in [('MLP', df_mlp), ('CNN', df_cnn)]:
    # V3_kc == V3_c
    for _, row in df.iterrows():
        c = int(row['class_c'])
        assert abs(row[f'V3_k{c}'] - row['V3_c']) < 1e-9, \
            f'{net}: V3_k{{c}} != V3_c at sample {row["sample_idx"]}, b={row["b"]}'
    # V2/V1 <= 1
    assert (df['V2_over_V1'] <= 1.0 + 1e-6).all(), f'{net}: V2 > V1 somewhere'
    # V1 constant per sample
    for idx, grp in df.groupby('sample_idx'):
        assert grp['V1'].nunique() == 1, f'{net}: V1 varies across b for sample {idx}'
    print(f'{net}: all sanity checks passed ✓')